# 01.3 — Data cleaning

Mục tiêu: đối chiếu lượng dữ liệu trước/sau cleaning và đọc sample đã release. Câu hỏi: vì sao không lọc mạnh code, command, path hoặc UI? Trả lời: đây là nội dung IT hợp lệ cần giữ cho Phase 02. Rule chính thức nằm trong `tools/build_phase01.py`.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'data/processed/it_en_vi').is_dir(): ROOT = ROOT.parent
report = json.loads((ROOT / 'data/processed/it_en_vi/dataset_report.json').read_text(encoding='utf-8'))
pd.Series(report['attrition'], name='pairs').to_frame()

,pairs
raw_rows,163869
after_source_cleaning,152634
after_global_exact_dedup,152044


In [2]:
per_source = pd.DataFrame(report['sources'])[['source_short_name', 'raw_rows', 'rows_after_cleaning']]
per_source.assign(removed=lambda x: x.raw_rows - x.rows_after_cleaning, kept_pct=lambda x: (100*x.rows_after_cleaning/x.raw_rows).round(2))

,source_short_name,raw_rows,rows_after_cleaning,removed,kept_pct
0,envitech_reasoning,15115,7104,8011,47.00
1,tech_viet_translation,100767,100458,309,99.69
2,gnome,149,148,1,99.33
3,ubuntu,5056,5036,20,99.60
4,kde4,42782,39888,2894,93.24


In [3]:
released = pd.read_json(ROOT / 'data/processed/it_en_vi/train.jsonl', lines=True)
released.loc[released.technical_tags.map(len).gt(0), ['en_clean', 'vi_clean', 'technical_tags']].sample(12, random_state=42)

,en_clean,vi_clean,technical_tags
64951,Enable advanced sharing to allow users to shar...,Hiệu lực khả năng chia sẻ cấp cao để cho các n...,[has_config]
15932,KGet has been temporarily disabled as download...,Trình KGet đã bị tắt tạm thời làm bộ quản lý v...,"[has_ui, has_config]"
14339,%1,Lỗi:% 1:% 2,[has_error]
39812,Study for a Python certification with this dis...,Học để lấy chứng chỉ Python với gói học trực t...,"[has_command, has_code_or_api]"
112601,This function is too tightly coupled with the ...,Hàm này quá gắn chặt với giao diện người dùng;...,"[has_ui, has_code_or_api]"
27287,& New,Lỗi lưu tập tin «% 1 » vào «% 2 ».,[has_error]
17113,Additional error protection for CD/DVD media,Bảo vệ chống lỗi trên CD/DVD,[has_error]
85340,The dashboard displays real-time metrics like ...,Bảng điều khiển hiển thị các chỉ số thời gian ...,[has_error]
60717,"A US state gave out 133,000 Chromebooks to peo...",Một bang của Mỹ phát miễn phí 133.000 Chromebo...,[has_version]
100400,Tecno Spark 10 Pro Moon Explorer Edition Annou...,Phiên bản Tecno Spark 10 Pro Moon Explorer đượ...,[has_version]


Kiểm tra bằng mắt ở cell trên để xác nhận technical text vẫn được giữ: `has_command`, `has_path`, `has_ui` hoặc `has_code_or_api` là nội dung IT hợp lệ, không phải lý do loại cặp câu.